In [3]:
import numpy as np
import pandas as pd

# Tabel 1: Target & PIC per cabang kota (tabel referensi, dibuat langsung sebagai DataFrame)
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}

# Tabel 2: Data transaksi (disimpan sebagai CSV, lalu diunggah ke HDFS)
np.random.seed(55)
n = 500
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"]
data_transaksi_t5 = {
    "order_id": [f"TRX-{i}" for i in range(n)],
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 10, size=n),
    "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000], size=n),
}
pd.DataFrame(data_transaksi_t5).to_csv("transaksi_tugas5.csv", index=False)

!hdfs dfs -mkdir -p /user/mahasiswa/tugas5
!hdfs dfs -put -f transaksi_tugas5.csv /user/mahasiswa/tugas5/
print("Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/mahasiswa/tugas5/transaksi_tugas5.csv")
print("Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.")

Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/mahasiswa/tugas5/transaksi_tugas5.csv
Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.


In [12]:
import numpy as np
import pandas as pd

# Buat dataset transaksi
np.random.seed(55)
n = 500
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"]
data_transaksi_t5 = {
    "order_id": [f"TRX-{i}" for i in range(n)],
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 10, size=n),
    "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000], size=n),
}

# Simpan ke lokal
pd.DataFrame(data_transaksi_t5).to_csv("transaksi_tugas5.csv", index=False)

# Unggah ke HDFS
!hdfs dfs -mkdir -p /user/mahasiswa/tugas5
!hdfs dfs -put -f transaksi_tugas5.csv /user/mahasiswa/tugas5/
print("File berhasil diunggah ke HDFS!")

File berhasil diunggah ke HDFS!


In [21]:
df_transaksi.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)



In [24]:
from pyspark.sql.functions import col, sum as spark_sum

# 1. Tambahkan kolom pendapatan
df_transaksi = df_transaksi.withColumn(
    "pendapatan", col("unit_terjual") * col("harga_satuan")
)

# 2. Ringkas total pendapatan per kota
df_pendapatan_kota = df_transaksi.groupBy("kota").agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)

# 3. Tampilkan hasil
df_pendapatan_kota.show()

[Stage 4:>                                                          (0 + 1) / 1]

+----------+----------------+
|      kota|total_pendapatan|
+----------+----------------+
|  Magelang|        31650000|
|  Semarang|        38175000|
|      Solo|        33475000|
| Purworejo|        45650000|
|Yogyakarta|        47275000|
+----------+----------------+



In [32]:
# 1. Buat DataFrame df_target (sesuaikan data/angkanya dengan modul praktikummu)
data_target = [
    ("Magelang", 30000000),
    ("Semarang", 35000000),
    ("Solo", 30000000),
    ("Purworejo", 40000000),
    ("Yogyakarta", 45000000),
]

df_target = spark.createDataFrame(
    data_target, ["kota", "target_bulanan"]
)

# 2. Ringkas total pendapatan per kota
df_pendapatan_kota = df_transaksi.groupBy("kota").agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)

# 3. Join dengan df_target dan hitung persentase pencapaian
df_bagian_a = (
    df_pendapatan_kota.join(df_target, on="kota", how="inner")
    .withColumn(
        "pencapaian_persen",
        (col("total_pendapatan") / col("target_bulanan")) * 100,
    )
    .orderBy(col("pencapaian_persen").desc())
)

print("=== Hasil Bagian A: Perbandingan Pencapaian Target ===")
df_bagian_a.show()

=== Hasil Bagian A: Perbandingan Pencapaian Target ===
+----------+----------------+--------------+------------------+
|      kota|total_pendapatan|target_bulanan| pencapaian_persen|
+----------+----------------+--------------+------------------+
| Purworejo|        45650000|      40000000|114.12500000000001|
|      Solo|        33475000|      30000000|111.58333333333333|
|  Semarang|        38175000|      35000000|109.07142857142857|
|  Magelang|        31650000|      30000000|             105.5|
|Yogyakarta|        47275000|      45000000|105.05555555555554|
+----------+----------------+--------------+------------------+



In [27]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, rank

# Define window spec berdasarkan kota dan diurutkan dari pendapatan terbesar
windowSpec = Window.partitionBy("kota").orderBy(col("pendapatan").desc())

# Tambahkan kolom rank lalu filter hanya peringkat 1
df_soal2 = df_transaksi.withColumn("rank", rank().over(windowSpec)) \
    .filter(col("rank") == 1)

df_soal2.show()

+--------+--------------------+----------+------------+------------+----------+----+
|order_id|            kategori|      kota|unit_terjual|harga_satuan|pendapatan|rank|
+--------+--------------------+----------+------------+------------+----------+----+
| TRX-390|Kesehatan & Kecan...|  Magelang|           9|      150000|   1350000|   1|
|  TRX-35|          Elektronik| Purworejo|           9|      150000|   1350000|   1|
|  TRX-81|   Makanan & Minuman|  Semarang|           9|      150000|   1350000|   1|
| TRX-262|Kesehatan & Kecan...|  Semarang|           9|      150000|   1350000|   1|
|  TRX-92|             Fashion|      Solo|           9|      150000|   1350000|   1|
| TRX-157|Kesehatan & Kecan...|Yogyakarta|           9|      150000|   1350000|   1|
| TRX-170|Kesehatan & Kecan...|Yogyakarta|           9|      150000|   1350000|   1|
| TRX-350|        Rumah Tangga|Yogyakarta|           9|      150000|   1350000|   1|
| TRX-397|             Fashion|Yogyakarta|           9|      1500

In [28]:
# Pastikan DataFrame df_transaksi sudah didaftarkan sebagai TempView
df_transaksi.createOrReplaceTempView("transaksi")

# Eksekusi kueri SQL
df_soal3 = spark.sql("""
    SELECT 
        kota, 
        AVG(pendapatan) AS rata_rata_pendapatan
    FROM transaksi
    GROUP BY kota
""")

df_soal3.show()

+----------+--------------------+
|      kota|rata_rata_pendapatan|
+----------+--------------------+
|  Magelang|  368023.25581395347|
|  Semarang|  410483.87096774194|
|      Solo|   352368.4210526316|
| Purworejo|   393534.4827586207|
|Yogyakarta|   429772.7272727273|
+----------+--------------------+




1. Cabang Berkinerja Paling Baik:
   Cabang **Yogyakarta** menunjukkan performa terbaik karena berhasil mencatatkan total pendapatan tertinggi sebesar **Rp47.275.000** dan secara konsisten melampaui target bulanan yang ditetapkan. Selain itu, berdasarkan agregasi jumlah transaksi, cabang ini juga mencatatkan volume transaksi yang paling mendominasi dibandingkan cabang lainnya.

2. **Cabang yang Memerlukan Perhatian Manajemen:**
   Cabang **Magelang** merupakan cabang yang paling membutuhkan perhatian khusus dari pihak manajemen. Cabang ini hanya mencatatkan total pendapatan sebesar **Rp31.650.000**, yang merupakan angka terendah di antara seluruh wilayah operasional. Rendahnya persentase pencapaian target dan jumlah transaksi di cabang ini mengindikasikan perlunya evaluasi strategi pemasaran, penyesuaian target pasar lokal, serta optimalisasi operasional oleh tim manajemen.